<div style="
    position: relative;
    width: 100%;
    height: 100%;
    overflow: hidden;
    border-radius: 10px;
">

<img src="../img/AML_Banner2.png" style="
    width: 100%;
    height: 100%;
    object-fit: cover;
">

<div style="
    position: absolute;
    top: 40%;
    left: 50%;
    transform: translate(-50%, -50%);
    color: white;
    font-size: 40px;
    font-weight: 600;
    text-align: center;
    font-family: Arial, sans-serif;
    text-shadow: 0px 3px 12px rgba(0,0,0,0.6);
">
Multi-class Classification Model</br>Evaluation
</div>

</div>

### EuroSAT
EuroSAT is a dataset of satellite images that helps researchers and engineers train artificial intelligence (AI) models to analyse land use and environmental changes from space.

It consists of thousands of high-resolution images captured by the European Space Agency's Sentinel-2 satellite, covering ten different land types such as forests, farmland, motorways, rivers, and urban areas.

This dataset is particularly useful for remote sensing, where AI can monitor deforestation, water bodies, and urban expansion over time. 

In agriculture, it helps farmers and policymakers track crop health, detect drought-affected regions, and optimise land use for better food production. 

In urban planning, EuroSAT assists governments in mapping city growth, identifying infrastructure needs, and managing natural resources more efficiently.

Models trained on EuroSAT, can help scientists develop automated systems to monitor environmental changes, improve disaster response, and make smarter decisions for sustainable development.

### Loading the data

First, we'll load the dataset. This is only a sample of the full dataset, with *only* 150 examples of each class. To achieve much better results, we would use the full dataset with approximately 3000 instances per class. Therefore, this is mainly for demonstration.

We will preprocess the images by resizing and normalising before training:

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# A function to preprocess our images (normalise)
def preprocess(image, label):
    image = image / 255.0
    return image, label

data_dir = "../data/EuroSAT"

dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="int",
    image_size=(64, 64),
    batch_size=32,
    shuffle=True,
    seed=7
)

# Extract class names to calculate the number of classes
class_names = dataset.class_names
num_classes = len(class_names)

print(f"Dataset loaded with {num_classes} classes:")
print(class_names)


We visualise a sample of the dataset by first shuffling the images, and then performing random sampling:

In [ ]:
import matplotlib.pyplot as plt

# Create a figure to hold all subplots
plt.figure(figsize=(12, 6))

# Take one batch from the dataset (images + labels)
for images, labels in dataset.take(1):
    
    # Loop through the first 18 images in the batch
    for i in range(18):
        
        # Create a 3x6 grid of subplots and select position i+1
        plt.subplot(3, 6, i + 1)

        # Convert tensor to NumPy array and normalise our pixel values to [0, 1]
        # This is because matplotlib expects floats in this range
        img = images[i].numpy() / 255.0  

        # Display the image for that subplot 
        plt.imshow(img)

        # Get the class index from the label tensor
        class_idx = int(labels[i].numpy())

        # Set the title using the class name
        plt.title(class_names[class_idx], fontsize=10)

        # Remove axis ticks and labels for a cleaner look
        plt.axis("off")

# Adjust spacing so plots and titles do not overlap
plt.tight_layout()

plt.show()

In the code above, we batch images to easily fetch multiple images at once, and convert the labels to readable class names for clearer visualisation, before plotting the samples.

We will define and train a neural network for classifcation of images based on multiple classes by way of a more advanced example.

First, let's extract train and test sets with 80% of the data used for training and 20% for testing:

In [ ]:
# Apply preprocessing (scale pixel values to 0-1)
dataset = dataset.map(preprocess)

# Get total number of batches
dataset_size = tf.data.experimental.cardinality(dataset).numpy()

# Use 67% of the data for training
train_size = int(0.67 * dataset_size)

# Take first part as training data
train_dataset = dataset.take(train_size)

# Use remaining data as test data
test_dataset = dataset.skip(train_size)

# Speed up training (store + prepare next batch)
train_dataset = train_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# Speed up testing
test_dataset = test_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

We'll use a *Convolutional Neural Network* (CNN), to demonstrate, which is a common choice for image classification. We will explore these models in more depth when we look at deep learning, but for now, the process is the same as we have seen before using classical machine learning models. A CNN is made up of several layers:

#### Convolutional layers
The CNN consists of three *convolutional layers*, these are special layers that act like filters scanning over the image to detect patterns such as edges, shapes, or textures, much like how our eyes might notice outlines or corners that detect patterns in the images.  The model processes our images (64x64 pixels, with 3 color channels) and learns patterns through these layers by way of feature extraction.

#### Max-pooling layers
This is followed by *max-pooling layers*, which shrink the image by taking only the most important information from small sections, imagine looking at a group of pixels and keeping only the brightest one to simplify the picture without losing key details. In short, this step reduces the image size while keeping important features.

#### Dense layers
The extracted information is then flattened to convert it from a 2D or 3D structure into a single long line of numbers, so it can be used by regular layers of a neural network. These are passed through *dense layers*, which are fully connected layers where every number influences the output, helping the network learn complex combinations and patterns. 

#### Drop out layer
This includes a *dropout layer*, a safety mechanism that randomly turns off some connections during training to prevent the model from relying too heavily on any one pattern. This helps avoid overfitting, so that the network does not simply memorise the training data, but instead can generalise to new inputs. 

#### Output layer
Finally, the *softmax output layer* turns the model's raw scores into probabilities, so it can assign probabilities to different classes of image in terms of whether an image is more likely representative of farm land, urban, or residential areas.

The model is compiled using the *Adam optimiser*, a method that adjusts how much the model changes during learning, which helps the model learn faster and more efficiently by balancing how quickly or slowly it updates its guesses. 

We also use *sparse categorical cross-entropy loss* as a way to measure how far off the model's predictions are from the correct answers when dealing with multiple categories; it works especially well when the correct answers are given as simple numbers like 0, 1, 2, etc. 

Training happens over 12 epochs, using a training dataset and validating against a test dataset. 

After training, the model is saved for future use:

In [ ]:
# Set up our CNN model
epochs_n = 12

model = keras.Sequential([
    keras.Input(shape=(64, 64, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

# Print the architecture
print(model.summary())

# Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Fit the model and retrieve the history for plotting later.
history = model.fit(train_dataset, validation_data=test_dataset, epochs=epochs_n)

# Save our model
model.save("model.keras")

Now the model has been trained, we can make predictions using our test image set. Let's plot a sample of test images with true and predicted labels to get a visual summary of how well the model is doing and what classes it might be struggling with:

In [ ]:
import numpy as np


# Create a large figure for displaying multiple images in a grid
plt.figure(figsize=(24, 11))  

# Get one batch of images and labels from the test dataset
for images, labels in test_dataset.take(1):
    # Predict class probabilities for each image in the batch
    preds = model.predict(images)

    # For each prediction, take the index of the highest probability as the predicted class
    pred_labels = np.argmax(preds, axis=1)

    # Loop through the first 12 images in the batch
    for i in range(12):
        # Set up a subplot: 3 rows x 6 columns
        plt.subplot(3, 6, i + 1)
        # Rescale image to display as raw image
        plt.imshow(images[i].numpy().clip(0, 1))
        
        # Get the true and predicted class names
        true_label = class_names[labels.numpy()[i]]
        predicted_label = class_names[pred_labels[i]]

        # Choose colour: green if prediction is correct, red otherwise
        if true_label == predicted_label:
            colour = "green"
        else: 
            colour = "red"

        # Display labels as title above the image
        plt.title(f"True: {true_label}\nPred: {predicted_label}", color=colour, fontsize=14)
        plt.axis('off')  # Hide axes to keep the layout clean

We will now evaluate the model on the whole test dataset and collect predictions for further analysis. We start by using `model.evaluate(test_dataset)`, which calculates the test *loss* and *accuracy*, providing an initial measure of performance. The test accuracy is then printed to give an immediate indication of how well the model generalises to unseen data.

In [ ]:
import numpy as np                              # For array operations
from sklearn.metrics import classification_report, confusion_matrix  # For evaluation metrics
import matplotlib.pyplot as plt                 # For plotting
import seaborn as sns                           # For heatmaps and advanced visuals

# Evaluate the trained model on the test dataset
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}")  # Display test accuracy (formatted to 4 decimal places)


Next, we iterate through the test dataset to extract actual labels (`y_true`) and predicted labels (`y_pred`). The model generates predictions in the form of class probability distributions, and `np.argmax(preds, axis=1)` selects the most probable class for each instance:

In [ ]:
# Prepare lists for true and predicted labels
y_true = []
y_pred = []

# Loop through test dataset and collect predictions
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)  # Predict class probabilities
    
    y_pred.extend(np.argmax(preds, axis=1))  # Get predicted class index
    y_true.extend(labels.numpy())            # Get true labels from tensor


The actual labels, initially stored as tensors, are converted to NumPy arrays using `labels.numpy()`. These predictions and true labels can now be used to compute our evaluation metrics such as *precision*, *recall*, and *confusion matrices* to gain deeper insights into the model's strengths and weaknesses.

### Confusion Matrices
We can generate a confusion matrix to compare the performance for each class. This provides insight into specific areas where the neural network might struggle, by inspecting the particular classes.

In a simple binary classification problem, a confusion matrix has four key parts: 
- *true positives (TP)* where the model correctly predicts the positive class, 
- *false positives (FP)* where it incorrectly predicts positive, 
- *false negatives (FN)* where it misses a positive case, and
- *true negatives (TN)* where it correctly predicts negative. 

For EuroSAT, which is a multi-class problem, the matrix is larger, but the idea is the same: each row shows the *actual* class and each column shows the *predicted* class. 

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 10))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("Confusion Matrix")

The diagonal entries are correct predictions (the equivalent of TP for each class), while the off-diagonal entries show mistakes. 

To interpret TP, FP, FN, and TN for a specific class, we treat it as a one-vs-all problem:  the diagonal cell is TP, the rest of that row are FN (missed predictions), the rest of that column are FP (incorrectly predicted as that class), and everything else in the matrix is TN.


### Accuracy
Let's now inspect the accuracy of our neural network model:

In [ ]:
# Evaluate the model
test_loss, test_acc = model.evaluate(test_dataset)

print(f"Test Accuracy: {test_acc:.4f}")

### Precision, Recall, and F1-Score
Next, we breakdown the models performance for all the classes. 

These measures are important in situations with imbalanced datasets, as accuracy alone might be misleading:

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

### ROC Curve and AUC
The ROC curve measures the trade-off between sensitivity (true positive rate) and specificity (1 - false positive rate). AUC indicates overall class separability (higher values = better model, 1 is perfect, 0.5 is random guessing). 

For imbalanced datasets, AUC gives more insight compared to accuracy alone.

ROC and AUC are commonly used for binary classification (deciding between two classes), they also work in multi-class settings by evaluating each class separately. 

In our case, since we have multiple classes, we calculate a separate ROC curve and AUC score for each one. This helps us pinpoint which specific classes the model is struggling with:

In [ ]:
import numpy as np                      # For working with arrays
import matplotlib.pyplot as plt         # For plotting graphs
from sklearn.metrics import roc_curve, auc            # ROC curve and AUC calculation
from sklearn.preprocessing import label_binarize      # Convert labels to one-hot encoded format
from itertools import cycle             # For cycling through colours in plots

# Prepare empty lists for true labels and predicted probabilities
y_true = []
y_pred_probs = []

# Loop through the test dataset to collect model predictions and actual labels
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)          # Get predicted probabilities for each class
    y_pred_probs.extend(preds)             # Add predicted probabilities to list
    y_true.extend(labels.numpy())          # Convert labels to NumPy arrays and add to list

# Convert collected data to NumPy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

# Convert true labels to one-hot encoding (required for ROC computation in multi-class)
num_classes = 10
y_true_bin = label_binarize(y_true, classes=range(num_classes))

# Initialise dictionaries to hold false positive rates, true positive rates, and AUCs
fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC curve and AUC for each class
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])  # ROC points
    roc_auc[i] = auc(fpr[i], tpr[i])  # Area Under Curve for each class

# Set up the plot
plt.figure(figsize=(10, 8))

# Cycle through a list of colours for each curve
colours = cycle(['blue', 'red', 'green', 'purple', 'orange', 'cyan', 'magenta', 'brown', 'olive', 'gray'])

# List of class names corresponding to each class index
class_names = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
               'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
               'River', 'SeaLake']

# Plot each class's ROC curve with its AUC value
for i, colour in zip(range(num_classes), colours):
    plt.plot(fpr[i], tpr[i], color=colour, label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

# Plot a dashed diagonal line to represent random chance (baseline)
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')

# Add axis labels and title
plt.xlabel('False Positive rate')
plt.ylabel('True Positive rate')
plt.title('Multi-class ROC curve for EuroSAT dataset')

# Show legend and grid
plt.legend(loc='lower right')
plt.grid(True)


Here we are analysing how our classification model behaves as we change the decision threshold used to convert predicted probabilities into class labels. Instead of fixing the threshold at a default value (such as 0.5), we sweep across all possible thresholds and measure how the model's performance changes.

At each threshold, we calculate sensitivity (true positive rate), to see how well the model detects actual positives, and specificity (true negative rate), to check how well it correctly rejects negatives. 

Let's plot these values against the threshold, so we can clearly see the trade-off between the two - increasing sensitivity often reduces specificity, and vice versa.


In [ ]:
# Choose a class to visualise
class_index = 2  # change this to any class (0–9)
print("Plot for class=", class_names[class_index])

# Recompute to also get thresholds
fpr_cls, tpr_cls, thresholds = roc_curve(
    y_true_bin[:, class_index],
    y_pred_probs[:, class_index]
)

# Compute specificity
specificity = 1 - fpr_cls

# Plot threshold vs sensitivity and specificity
plt.figure(figsize=(10, 6))

plt.plot(thresholds, tpr_cls, label="Sensitivity (Detecting positives correctly)")
plt.plot(thresholds, specificity, label="Specificity (Rejecting negatives correctly)")

plt.xlabel("Decision Threshold (classification cutoff)")
plt.ylabel("Performance Score")
plt.title(f"Model Performance ({class_names[class_index]})")

plt.legend( bbox_to_anchor=(0.55, .90))
plt.grid(True)

plt.show()

This is important because different applications require different balances. For example, in medical diagnosis we may prioritise sensitivity to avoid missing a disease, whereas in spam detection we may prioritise specificity to avoid flagging legitimate emails. 

Looking at this trade-off allows us to choose a threshold that aligns with the real-world cost of errors, rather than relying on an arbitrary default.

### Loss Function
After training the model, we can look at the following data to see how well the learning process went:

- `history.history['loss']`: </br>
This shows how much error the model made on the *training set* (the data it learnt from) after each round of learning, called an *epoch*.  
- `history.history['val_loss']`:  </br>
This shows the error on the *validation set* based on separate data the model hasn't seen before, used to test how well it's generalising during training.

Ideally, both the training and validation losses should go down over time indicating that the model is improving and learning useful patterns.

However, if the *validation loss* starts to go up while the *training loss* keeps going down, it might mean the model is *overfitting*. In other words, the model is memorising the training data too well and struggling to perform well on new, unseen data:

In [ ]:
import matplotlib.pyplot as plt

# Plot the training and validation loss
plt.figure(figsize=(10, 4))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Training and Validation loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

plt.grid(True)


### What have we learnt?
In this notebook, we explored how to evaluate a convolutional neural network on a multi-class image classification task using the EuroSAT dataset. We moved beyond simply measuring accuracy and looked at how predictions are distributed across classes using tools like the confusion matrix. This helped us understand not just how often the model is correct, but where it makes mistakes, revealing patterns such as confusion between visually similar land types. 

We also reinforced the importance of preprocessing, such as normalising image data, to ensure models train and visualise correctly.

We then focused on interpreting model performance in a more detailed way by breaking evaluation down per class. When we treat each class as a one-vs-all problem, we are able to define true positives, false positives, false negatives, and true negatives for each category. This allowed us to connect the confusion matrix to more advanced metrics like precision and recall, giving a clearer picture of model strengths and weaknesses. 

Overall, the key takeaway is that evaluation in multi-class problems requires looking beyond a single metric and carefully analysing how predictions are distributed across all classes.